# minicompiler: GPU validation on Colab

Rebuilds minicompiler with its CUDA backend on a Colab GPU and records the same benchmarks as the RTX 3060 Laptop results in the repository, so every GPU claim can be checked on a second GPU:

1. prints the GPU, driver, CUDA and PyTorch versions;
2. clones the repository at a pinned commit;
3. builds with the CUDA backend for this GPU's compute capability (Eigen and GoogleTest are downloaded by CMake when missing);
4. runs every test, and fails if any GPU test is skipped;
5. runs `scripts/run_gpu_benchmarks.sh`: fused vs unfused elementwise kernels, every FP32 matmul kernel vs cuBLAS, the tensor-core kernels vs cuBLAS with the same input format (on a T4, compute capability 7.5, only FP16 runs: TF32 and BF16 tensor cores need 8.0), and the CUDA backend vs PyTorch eager and `torch.compile`, each repeated `RUNS` times and written to `results/gpu/*.json` with the commit hash;
6. shows the result tables and downloads the JSON files.

**To run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all. It takes roughly 35–50 minutes on a T4, most of it `torch.compile` and the timing runs.

In [ ]:
import os

REPO = "https://github.com/kartsen03/minicompiler.git"
COMMIT = "b4cc679"  # the commit to validate
RUNS = 3              # repetitions of the whole benchmark sequence

os.environ.update(REPO=REPO, COMMIT=COMMIT, RUNS=str(RUNS))
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
os.environ["MINICOMPILER_HOST_POWER_PLAN"] = "n/a (Colab VM)"

In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total,clocks.max.sm --format=csv
nvcc --version | tail -2
cmake --version | head -1
python3 -c "import torch, triton; print('PyTorch', torch.__version__, '| its CUDA', torch.version.cuda, '| Triton', triton.__version__, '|', torch.cuda.get_device_name(0))"

In [ ]:
%%bash
set -eo pipefail
rm -rf minicompiler build
git clone -q "$REPO" minicompiler
git -C minicompiler checkout -q "$COMMIT"
git -C minicompiler log --oneline -1

In [ ]:
%%bash
set -eo pipefail
# Ahead-of-time kernels (the matmuls) for this GPU; the elementwise kernels are
# compiled at run time by NVRTC for whatever GPU is present.
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
cmake -S minicompiler -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES="$CC" | grep "minicompiler:"
cmake --build build -j "$(nproc)" > build.log 2>&1 || { tail -40 build.log; exit 1; }
grep -iE "warning|error" build.log || echo "built with no warnings"

In [ ]:
%%bash
set -eo pipefail
cd build
ctest -j 2 --output-on-failure | tail -3
# The GPU tests skip themselves when no GPU is visible; here none may.
tests/minicompiler_tests --gtest_filter='CudaBackend.*' > cuda_tests.txt || true
grep -E "^\[ +(OK|SKIPPED|FAILED) +\]" cuda_tests.txt
if grep -qE "^\[ +(SKIPPED|FAILED) +\]" cuda_tests.txt; then echo "GPU tests did not all pass"; exit 1; fi

In [ ]:
%%bash
set -eo pipefail
cd minicompiler
BUILD_DIR=../build PYTHON=python3 RUNS="$RUNS" scripts/run_gpu_benchmarks.sh 2>&1 | grep -v "max_autotune_gemm"
ls -l results/gpu

In [ ]:
import subprocess
from IPython.display import Markdown, display

tables = subprocess.run(["python3", "scripts/results_tables.py", "--gpu", "results/gpu"], cwd="minicompiler",
                        capture_output=True, text=True, check=True).stdout
display(Markdown(tables))

In [ ]:
import json
import shutil

from google.colab import files

commit = subprocess.check_output(["git", "-C", "minicompiler", "rev-parse", "--short", "HEAD"], text=True).strip()
gpu = json.load(open("minicompiler/results/gpu/matmul.json"))["device"]["name"].replace(" ", "_")
archive = shutil.make_archive(f"minicompiler_gpu_results_{gpu}_{commit}", "zip", "minicompiler/results", "gpu")
print("results recorded at commit", commit, "->", archive)
files.download(archive)